In [1]:
import os
import sys
import json
import time
import random
from dataclasses import dataclass, asdict
from typing import Dict, Any

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.notebook import tqdm

PROJECT_ROOT = "/autodl-fs/data/archive"
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from models.dataset import (
    build_scaler_streaming,
    YelpBertDataset,
)
from models.bert_cross_attention import BERTCrossAttentionClassifier


@dataclass
class Args:
    task: str = "sentiment"
    train_path: str = "data/splits_small/train.jsonl"
    val_path: str = "data/splits_small/val.jsonl"
    test_path: str = "data/splits_small/test.jsonl"
    output_dir: str = "outputs/checkpoints/bert_cross_attention"

    pretrained_model_name: str = "/autodl-fs/data/archive/train/bert-base-uncased"
    text_field: str = "text"
    max_length: int = 128
    num_meta_tokens: int = 4
    num_heads: int = 4
    dropout: float = 0.3
    freeze_bert: bool = True

    batch_size: int = 8
    epochs: int = 2
    lr: float = 2e-5
    weight_decay: float = 1e-2
    warmup_ratio: float = 0.1
    grad_clip: float = 1.0
    seed: int = 42

    num_workers: int = 0
    pin_memory: bool = False


args = Args()


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_num_classes(task: str):
    if task == "sentiment":
        return 3
    if task == "rating":
        return 9
    raise ValueError(f"Unsupported task: {task}")


def save_json(obj: Dict[str, Any], path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def bert_collate_with_meta(batch, pad_token_id=0):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels, meta_features = [], [], [], []

    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(torch.cat([x["input_ids"], torch.full((pad_len,), pad_token_id, dtype=torch.long)]))
        attention_mask.append(torch.cat([x["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(x["label"])
        meta_features.append(x["meta_features"])

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_mask),
        "labels": torch.stack(labels),
        "meta_features": torch.stack(meta_features),
    }


def train_one_epoch(model, dataloader, optimizer, scheduler, criterion, device, grad_clip, epoch):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(dataloader, desc=f"Epoch {epoch:02d} Training", dynamic_ncols=True, leave=True)
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        meta_features = batch["meta_features"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask, meta_features=meta_features)
        loss = criterion(logits, labels)
        loss.backward()

        if grad_clip is not None and grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.detach().cpu().tolist())
        all_labels.extend(labels.detach().cpu().tolist())

        running_loss = total_loss / max(1, len(all_labels))
        running_acc = accuracy_score(all_labels, all_preds)
        pbar.set_postfix({"loss": f"{running_loss:.4f}", "acc": f"{running_acc:.4f}"})

    return {
        "loss": total_loss / max(1, len(all_labels)),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
    }


def evaluate(model, dataloader, criterion, device, desc="Evaluating"):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc=desc, dynamic_ncols=True, leave=True)
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            meta_features = batch["meta_features"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask, meta_features=meta_features)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

            running_loss = total_loss / max(1, len(all_labels))
            running_acc = accuracy_score(all_labels, all_preds)
            pbar.set_postfix({"loss": f"{running_loss:.4f}", "acc": f"{running_acc:.4f}"})

    return {
        "loss": total_loss / max(1, len(all_labels)),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "predictions": all_preds,
        "labels": all_labels,
    }


def main(args: Args):
    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] Device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(args.pretrained_model_name)
    scaler = build_scaler_streaming(args.train_path)
    save_json(scaler.state_dict(), os.path.join(args.output_dir, "scaler.json"))

    train_dataset = YelpBertDataset(args.train_path, args.task, tokenizer, args.max_length, args.text_field, True, scaler)
    val_dataset = YelpBertDataset(args.val_path, args.task, tokenizer, args.max_length, args.text_field, True, scaler)
    test_dataset = YelpBertDataset(args.test_path, args.task, tokenizer, args.max_length, args.text_field, True, scaler)

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers, pin_memory=args.pin_memory, collate_fn=lambda b: bert_collate_with_meta(b, tokenizer.pad_token_id))
    val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers, pin_memory=args.pin_memory, collate_fn=lambda b: bert_collate_with_meta(b, tokenizer.pad_token_id))
    test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers, pin_memory=args.pin_memory, collate_fn=lambda b: bert_collate_with_meta(b, tokenizer.pad_token_id))

    meta_dim = train_dataset[0]["meta_features"].shape[0]
    model = BERTCrossAttentionClassifier(
        pretrained_model_name=args.pretrained_model_name,
        num_classes=get_num_classes(args.task),
        meta_dim=meta_dim,
        num_meta_tokens=args.num_meta_tokens,
        dropout=args.dropout,
        num_heads=args.num_heads,
        freeze_bert=args.freeze_bert,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

    total_steps = len(train_loader) * args.epochs
    warmup_steps = int(total_steps * args.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_val_f1 = -1.0
    best_model_path = os.path.join(args.output_dir, "best_model.pt")
    history = []

    epoch_bar = tqdm(range(1, args.epochs + 1), desc="Epochs", dynamic_ncols=True)
    for epoch in epoch_bar:
        start_time = time.time()

        train_metrics = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device, args.grad_clip, epoch)
        val_metrics = evaluate(model, val_loader, criterion, device, desc=f"Epoch {epoch:02d} Validation")

        elapsed = time.time() - start_time
        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "elapsed_sec": elapsed,
        })

        epoch_bar.set_postfix({"train_f1": f"{train_metrics['macro_f1']:.4f}", "val_f1": f"{val_metrics['macro_f1']:.4f}"})

        print(
            f"[Epoch {epoch:02d}] "
            f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['accuracy']:.4f} train_f1={train_metrics['macro_f1']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} val_f1={val_metrics['macro_f1']:.4f} | "
            f"time={elapsed:.2f}s"
        )

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            torch.save(model.state_dict(), best_model_path)
            print(f"[INFO] Best model saved to: {best_model_path}")

    save_json(asdict(args), os.path.join(args.output_dir, "config.json"))
    save_json({"history": history}, os.path.join(args.output_dir, "history.json"))

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    test_metrics = evaluate(model, test_loader, criterion, device, desc="Testing")

    print("\n========== Final Test Results ==========")
    print(f"Test Loss: {test_metrics['loss']:.4f}")
    print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"Test Macro-F1: {test_metrics['macro_f1']:.4f}")

    report = classification_report(test_metrics["labels"], test_metrics["predictions"], digits=4)
    print(report)

    save_json(
        {
            "test_loss": test_metrics["loss"],
            "test_accuracy": test_metrics["accuracy"],
            "test_macro_f1": test_metrics["macro_f1"],
            "classification_report": report,
        },
        os.path.join(args.output_dir, "test_metrics.json"),
    )

args.task = "rating"
args.output_dir = f"outputs/checkpoints/bert_cross_attention_{args.task}"
main(args)

[INFO] Device: cuda


Building scaler from train.jsonl: 0it [00:00, ?it/s]

Loading train.jsonl: 0it [00:00, ?it/s]

Loading val.jsonl: 0it [00:00, ?it/s]

Loading test.jsonl: 0it [00:00, ?it/s]

...

Epoch 02 Training:   0%|          | 0/125000 [00:00<?, ?it/s]

Epoch 02 Validation:   0%|          | 0/25000 [00:00<?, ?it/s]

[Epoch 02] train_loss=0.7567 train_acc=0.6773 train_f1=0.5553 | val_loss=0.7223 val_acc=0.6908 val_f1=0.5710 | time=15367.14s
[INFO] Best model saved to: outputs/checkpoints/bert_cross_attention_rating/best_model.pt


Testing:   0%|          | 0/25000 [00:00<?, ?it/s]


========== Final Test Results ==========
Test Loss: 0.7172
Test Accuracy: 0.6933
Test Macro-F1: 0.5732
              precision    recall  f1-score   support

           0     0.7424    0.8612    0.7974     30466
           2     0.4338    0.3078    0.3601     15800
           4     0.4684    0.3187    0.3793     19800
           6     0.5164    0.4663    0.4901     41145
           8     0.7980    0.8843    0.8390     92789

    accuracy                         0.6933    200000
   macro avg     0.5918    0.5677    0.5732    200000
weighted avg     0.6702    0.6933    0.6775    200000

